# Silver — Exchange Rates & OMX
Cleans market data, derives ISK/EUR cross-rates, and renames columns to clean business names.

**Source:** `bronze.yahoo_finance_raw`  
**Output:** `silver.exchange_rates` (Delta table)  
**Key derivation:** `iskeur_close = iskusd_close / eurusd_close`

In [ ]:
df_raw = spark.read.table("bronze.yahoo_finance_raw")
df_raw.createOrReplaceTempView("v_fx_raw")

In [ ]:
df_silver = spark.sql("""
    WITH raw_staged AS (
        SELECT
            date,
            CAST(close_eurusd_x AS DOUBLE) AS eurusd_raw,
            CAST(close_iskusd_x AS DOUBLE) AS iskusd_raw,
            CAST(close_omx     AS DOUBLE) AS omx_raw,
            ingested_at
        FROM v_fx_raw
    ),
    derived_metrics AS (
        SELECT
            date,
            eurusd_raw AS eurusd_close,
            -- If it's a spike, set to NULL so the MERGE can overwrite the old bad data
            CASE WHEN iskusd_raw < 0.02 THEN iskusd_raw ELSE NULL END AS iskusd_close,
            omx_raw AS omx_close,
            CASE 
                WHEN eurusd_raw > 0 THEN 
                    CASE WHEN (iskusd_raw / eurusd_raw) < 0.02 THEN ROUND(iskusd_raw / eurusd_raw, 6) ELSE NULL END
                ELSE NULL 
            END AS iskeur_close,
            ingested_at
        FROM raw_staged
        WHERE date IS NOT NULL
        -- Note: We DON'T filter the whole row out here, so the MERGE finds the date to fix it
    ),
    deduplicated AS (
        SELECT 
            date, eurusd_close, iskusd_close, omx_close, iskeur_close, ingested_at,
            ROW_NUMBER() OVER (PARTITION BY date ORDER BY ingested_at DESC) as rn
        FROM derived_metrics
    )
    SELECT 
        date, eurusd_close, iskusd_close, omx_close, iskeur_close,
        CURRENT_TIMESTAMP() AS processed_at
    FROM deduplicated
    WHERE rn = 1
""")

df_silver.createOrReplaceTempView("v_fx_silver_staged")

In [ ]:
if not spark.catalog.tableExists("silver.exchange_rates"):
    df_silver.write.format("delta").saveAsTable("silver.exchange_rates")
    print("Created silver.exchange_rates.")
else:
    spark.sql("""
        MERGE INTO silver.exchange_rates AS target
        USING v_fx_silver_staged AS source
        ON target.date = source.date
        WHEN MATCHED THEN UPDATE SET
            target.eurusd_close = source.eurusd_close,
            target.iskusd_close = source.iskusd_close,
            target.omx_close    = source.omx_close,
            target.iskeur_close = source.iskeur_close,
            target.processed_at = source.processed_at
        WHEN NOT MATCHED THEN INSERT (date, eurusd_close, iskusd_close, omx_close, iskeur_close, processed_at)
        VALUES (source.date, source.eurusd_close, source.iskusd_close, source.omx_close, source.iskeur_close, source.processed_at)
    """)

print(f"Merge complete. Final count: {spark.table('silver.exchange_rates').count()}")